# NemotronLabs RE-USE laughter on Kaggle

Attach your archival WAV as a private input dataset. In the private notebook, select **GPU T4 x2**, enable **Internet**, then choose **Save Version → Save & Run All** for background execution. Download both audio files from the saved version's Output tab.


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

def run(*args, **kwargs):
    subprocess.run([str(arg) for arg in args], check=True, **kwargs)

working = Path("/kaggle/working")
project = working / "eileen-chang-clean-up"
revision = "71667393c8b7f03506d061c153739dbb057d387c"

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "Kaggle did not expose an NVIDIA GPU to this run. "
        "Open the notebook editor and select GPU T4 x2 under Settings > Accelerator. "
        "If the option is disabled, check Phone Verification and your GPU quota."
    )
run("nvidia-smi")
for tool in ("ffmpeg", "ffprobe", "git"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"{tool} is missing from this Kaggle image.")

if not project.exists():
    run("git", "clone", "https://github.com/jianershi/eileen-chang-clean-up.git", project)
run("git", "checkout", "--detach", revision, cwd=project)
run(sys.executable, project / "scripts/setup_gpu.py", "--install-torch", cwd=project)


In [ ]:
import re

matches = list(Path("/kaggle/input").rglob("RA-T.106_nd_Reyher_master.wav"))
if len(matches) != 1:
    raise RuntimeError("Attach the private dataset containing one copy of the archival WAV.")

# Select the exact laughter checkpoint used in the reported Kaggle run.
profile = project / "src/eileen_cleanup/profile.py"
code = profile.read_text()
for key, value in {
    "MODEL_REPOSITORY": "nvidia/NVIDIA-NemotronLabs-RE-USE-laughter",
    "MODEL_REVISION": "1b1521dd659608e1e063ada83e0acfe1da473a8b",
    "MODEL_WEIGHTS_SHA256": "35c2a4e284515275d2e49a8e8cb5cd392ae67a1f1d345de1c12a0202c7b6b610",
    "OUTPUT_STEM": "NVIDIA_NemotronLabs_REUSE_laughter_full_recording"
}.items():
    code, count = re.subn(rf"^{key} = .*$", f'{key} = "{value}"', code, flags=re.M)
    if count != 1:
        raise RuntimeError(f"Could not update {key} in profile.py")
profile.write_text(code)

# The laughter checkpoint nests its configuration under cfg.
loader = project / "src/eileen_cleanup/model.py"
loader_code = loader.read_text()
config_line = '        cfg = json.loads((directory / "config.json").read_text())'
unwrap_line = '        cfg = cfg.get("cfg", cfg)'
if unwrap_line not in loader_code:
    if loader_code.count(config_line) != 1:
        raise RuntimeError("Could not find the config loader in model.py")
    loader.write_text(loader_code.replace(config_line, config_line + "\n" + unwrap_line, 1))

output = working / "reuse_laughter_outputs"
run(sys.executable, "-m", "eileen_cleanup", "restore", matches[0],
    "--work-dir", working / "reuse_laughter_work", "--output-dir", output,
    "--expected-duration", "4365.67", cwd=project)
